# Cuaderno auxiliar: capítulo 1, controles del 25 de octubre e inventario

Ubicación: `C:\SERCOP_FINAL\02_codigo\`. Reúne las celdas que salieron de `plan_A\cuaderno19.ipynb` el 25 de octubre de 2026 (para que ese cuaderno quede solo con el análisis de H3) y las celdas de control de ese día. Todas leen; las únicas que escriben archivos lo dicen en su encabezado. Ninguna imprime RUC.

**No uses "Ejecutar todo".** Varias celdas se corren una sola vez y se detienen a propósito si su archivo ya existe, y la versión anterior del capítulo 1 falla a propósito (se conserva como evidencia). Corre cada celda por separado.

| Sección | Qué tiene | ¿Se vuelve a correr? |
|---|---|---|
| A | Capítulo 1: comprobación de columnas y tablas y figuras corregidas | Sí, cuando haga falta regenerar `capitulo1\` |
| B | Controles del 25 de octubre: huella del cuaderno 19, manifiesto para la tutora y reconstrucción de la concurrencia | La huella y el manifiesto, no (se detienen si su archivo existe). La reconstrucción, sí |
| C | Verificación de la huella del cuaderno 19 | Sí: antes de correr la celda 3 de H3 con las hojas reales |
| D | Inventario y uniones del 20 de octubre | Sí, si se quiere repetir el inventario |
| E | Versión anterior del capítulo 1 | No: la primera celda falla (`BinderException`) y la reemplaza la sección A |

Copia exacta del cuaderno 19 antes de limpiarlo: `plan_A\_versiones\cuaderno19_antes_de_limpiar_2026-10-25.ipynb`.

## A. Capítulo 1 (vigente)

Primero la comprobación de columnas y después la celda corregida, que escribe en `C:\SERCOP_FINAL\capitulo1\` las tablas A y B, las figuras A y B y `LEEME_universo.txt`.

In [ ]:
# =====================================================================================
# CAPÍTULO 1 · COMPROBACIÓN DE COLUMNAS (solo lectura, solo nombres y conteos, ningún RUC)
#   Dice qué columnas de familia y de año tiene planA_procesos_modelo y si el año guardado
#   (anio_proceso) coincide con el año de inicio del período de ofertas (fecha_inicio_tender).
# =====================================================================================
import duckdb
PM = r"C:\SERCOP_FINAL\plan_A\planA_procesos_modelo.parquet".replace("\\", "/")
con = duckdb.connect()
tipos = {r[0]: r[1] for r in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{PM}')").fetchall()}
for c in ["familia", "familia_procedimiento", "anio_proceso", "fecha_inicio_tender", "monto_final"]:
    print(f"  {c:<22} {'existe, tipo ' + tipos[c] if c in tipos else 'NO EXISTE'}")
print("  otras columnas con 'familia' o 'anio' en el nombre:",
      [c for c in tipos if ("familia" in c or "anio" in c) and c not in ("familia", "familia_procedimiento", "anio_proceso")])

if "anio_proceso" in tipos and "fecha_inicio_tender" in tipos:
    f = ("year(TRY_CAST(substr(fecha_inicio_tender, 1, 10) AS DATE))" if "VARCHAR" in tipos["fecha_inicio_tender"]
         else "year(fecha_inicio_tender)")
    a = "TRY_CAST(anio_proceso AS INTEGER)"
    r = con.execute(f"""
        SELECT COUNT(*), COUNT(*) FILTER (WHERE {a} IS NULL), COUNT(*) FILTER (WHERE {f} IS NULL),
               COUNT(*) FILTER (WHERE {a} IS DISTINCT FROM {f})
        FROM read_parquet('{PM}')""").fetchone()
    print(f"\nUniverso: todas las filas de planA_procesos_modelo ({r[0]:,})")
    print(f"  anio_proceso nulo: {r[1]:,} | fecha_inicio_tender nula: {r[2]:,} | "
          f"filas donde anio_proceso es distinto del año de fecha_inicio_tender: {r[3]:,}")
    if r[3]:
        print("\n  Las 15 combinaciones más frecuentes donde difieren (anio_proceso, año de inicio, procesos):")
        print(con.execute(f"""
            SELECT {a} AS anio_proceso, {f} AS anio_inicio, COUNT(*) AS procesos
            FROM read_parquet('{PM}') WHERE {a} IS DISTINCT FROM {f}
            GROUP BY 1, 2 ORDER BY procesos DESC LIMIT 15""").df().to_string(index=False))

In [ ]:
# =====================================================================================
# CAPÍTULO 1 (versión corregida del 25 de octubre): tablas y figuras de contexto
#   Solo lectura de la base; sin puntajes, sin SRI y sin ningún RUC. Reemplaza a las dos celdas
#   anteriores del capítulo 1 (la primera fallaba porque la columna se llama familia_procedimiento).
#   Escribe en C:\SERCOP_FINAL\capitulo1 las mismas dos tablas y dos figuras, más un LEEME con el
#   universo de cada cifra.
# =====================================================================================
import os, duckdb, pandas as pd, matplotlib.pyplot as plt
from datetime import datetime

BASE = r"C:\SERCOP_FINAL\plan_A"
OUT = r"C:\SERCOP_FINAL\capitulo1"
PM = os.path.join(BASE, "planA_procesos_modelo.parquet").replace("\\", "/")
IND = os.path.join(BASE, "planA_indicadores.parquet").replace("\\", "/")
ANIO = "year(fecha_inicio_tender)"            # regla de la tesis: año = inicio del período de ofertas
con = duckdb.connect()

# 0) Control: la familia de esta tabla debe ser la misma que usan los modelos (planA_indicadores.familia)
cruce = con.execute(f"""
    SELECT p.familia_procedimiento, i.familia AS familia_modelos, COUNT(*) AS procesos
    FROM read_parquet('{PM}') p JOIN read_parquet('{IND}') i USING (ocid)
    GROUP BY 1, 2 ORDER BY procesos DESC""").df()
n_base = con.execute(f"SELECT COUNT(*) FROM read_parquet('{PM}')").fetchone()[0]
print(f"Base: {n_base:,} procesos | unidos con los indicadores por ocid: {int(cruce['procesos'].sum()):,}")
print("Familia de la base frente a la familia de los modelos:")
print(cruce.to_string(index=False))
uno_a_uno = (len(cruce) == cruce["familia_procedimiento"].nunique() == cruce["familia_modelos"].nunique()
             and int(cruce["procesos"].sum()) == n_base and cruce.notna().all().all())
if not uno_a_uno:
    raise SystemExit("La familia de la base no corresponde una a una con la de los modelos: no escribo nada. Pégame esta salida.")
print("Correspondencia uno a uno: la tabla B usa las mismas familias que los modelos.\n")

# 1) Procesos y monto adjudicado por año (2015-2024)
por_anio = con.execute(f"""
    SELECT {ANIO} AS anio, COUNT(*) AS procesos, SUM(monto_final)/1e6 AS millones_usd
    FROM read_parquet('{PM}') WHERE {ANIO} BETWEEN 2015 AND 2024
    GROUP BY 1 ORDER BY 1""").df()
fuera = con.execute(f"""
    SELECT {ANIO} AS anio, COUNT(*) AS procesos, SUM(monto_final)/1e6 AS millones_usd
    FROM read_parquet('{PM}') WHERE {ANIO} NOT BETWEEN 2015 AND 2024 OR {ANIO} IS NULL
    GROUP BY 1 ORDER BY 1""").df()
total = con.execute(f"SELECT COUNT(*), SUM(monto_final)/1e6 FROM read_parquet('{PM}')").fetchone()
print(f"Base completa: {total[0]:,} procesos | {total[1]:,.1f} millones USD")
print(f"\nTabla A. Procesos y monto adjudicado por año de inicio, 2015-2024 "
      f"({int(por_anio['procesos'].sum()):,} procesos de {total[0]:,})")
print(por_anio.round(1).to_string(index=False))
print("Quedan fuera de la tabla A (años de inicio fuera de 2015-2024):")
print(fuera.round(1).to_string(index=False))
if int(por_anio["procesos"].sum()) + int(fuera["procesos"].sum()) != total[0]:
    raise SystemExit("La tabla A y los excluidos no suman la base completa: no escribo nada.")

# 2) Procesos y monto por familia de procedimiento (base completa, 2015-2026)
por_familia = con.execute(f"""
    SELECT familia_procedimiento AS familia, COUNT(*) AS procesos, SUM(monto_final)/1e6 AS millones_usd,
           MEDIAN(monto_final) AS monto_mediano_usd
    FROM read_parquet('{PM}') GROUP BY 1 ORDER BY millones_usd DESC""").df()
por_familia["pct_monto"] = 100 * por_familia["millones_usd"] / por_familia["millones_usd"].sum()
print(f"\nTabla B. Procesos y monto por familia de procedimiento, base completa "
      f"({int(por_familia['procesos'].sum()):,} procesos)")
print(por_familia.round(1).to_string(index=False))

# 3) Archivos (se reescriben las mismas tablas y figuras; se agrega el LEEME)
os.makedirs(OUT, exist_ok=True)
por_anio.to_csv(os.path.join(OUT, "tabla_A_por_anio.csv"), index=False, sep=";", decimal=",")
por_familia.to_csv(os.path.join(OUT, "tabla_B_por_familia.csv"), index=False, sep=";", decimal=",")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(por_anio["anio"].astype(str), por_anio["procesos"], color="#9aa5b1")
ax.set_ylabel("Procesos")
ax2 = ax.twinx()
ax2.plot(por_anio["anio"].astype(str), por_anio["millones_usd"], color="#1f5fbf", marker="o")
ax2.set_ylabel("Monto adjudicado (millones USD)")
ax.set_title("Procesos y monto adjudicado por año, 2015-2024")
ax.spines["top"].set_visible(False); ax2.spines["top"].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "figura_A_por_anio.png"), dpi=300); plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 4.5))
t = por_familia.sort_values("millones_usd")
ax.barh(t["familia"].astype(str), t["millones_usd"], color="#9aa5b1")
for y, v in enumerate(t["millones_usd"]):
    ax.text(v, y, f" {v:,.0f}", va="center", fontsize=8)
ax.set_xlabel("Monto adjudicado (millones USD)")
ax.set_title("Monto adjudicado por familia de procedimiento, 2015-2026")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "figura_B_por_familia.png"), dpi=300); plt.close(fig)

mil = lambda n: f"{int(n):,}".replace(",", ".")          # miles con punto, como en la tesis
f_txt = lambda d: "; ".join(f"{int(r.anio) if pd.notna(r.anio) else 'sin año'}: {mil(r.procesos)} procesos" for r in d.itertuples())
with open(os.path.join(OUT, "LEEME_universo.txt"), "w", encoding="utf-8") as f:
    f.write(f"Generado el {datetime.now():%Y-%m-%d %H:%M} desde planA_procesos_modelo.parquet (solo lectura).\n\n"
            f"Universo de la base: {mil(total[0])} procesos sin catálogo electrónico, con RUC de proveedor y monto "
            f"positivo, 2015 a 2026 (2026 incompleto: corte del 5 de septiembre de 2026). "
            f"Monto = monto_final (suma de las adjudicaciones), en millones de USD corrientes.\n"
            f"Tabla A y figura A: año = year(fecha_inicio_tender), inicio del período de ofertas; años 2015 a 2024, "
            f"{mil(por_anio['procesos'].sum())} procesos. Fuera de la serie: {f_txt(fuera)}.\n"
            f"Tabla B y figura B: base completa, {mil(por_familia['procesos'].sum())} procesos, por familia de "
            f"procedimiento (familia_procedimiento, la misma de los modelos).\n")
print("\nTablas, figuras y LEEME_universo.txt guardados en", OUT)

## B. Controles del 25 de octubre

Se corrieron una vez el 25 de octubre; sus salidas quedan como evidencia.

- **Huella del cuaderno 19:** escribió `preregistro\huella_cuaderno19.json`. **No volver a correr:** si el archivo existe, se detiene.
- **Manifiesto para la tutora:** escribió `plan_A\documentacion\manifiesto_2026-10-25.csv`. **No volver a correr:** si el archivo existe, se detiene.
- **Reconstrucción de la concurrencia y de las columnas `_1`:** solo lee; se puede repetir.

In [ ]:
# =====================================================================================
# HUELLA DEL CUADERNO 19 ANTES DE RECIBIR LAS HOJAS (correr UNA sola vez)
#   Córrela desde aux_capitulo1_e_inventario.ipynb, no desde el cuaderno 19, con
#   cuaderno19.ipynb guardado y cerrado. Solo lee el cuaderno 19 y escribe un archivo nuevo:
#   C:\SERCOP_FINAL\preregistro\huella_cuaderno19.json. Si ese archivo ya existe, se detiene.
# =====================================================================================
import os, json, hashlib, platform
from datetime import datetime

SERCOP = r"C:\SERCOP_FINAL"
CUAD = os.path.join(SERCOP, "plan_A", "cuaderno19.ipynb")
DEST = os.path.join(SERCOP, "preregistro", "huella_cuaderno19.json")
LLENAS = os.path.join(SERCOP, "revision_h3_llenas")

def sha_bytes(b):
    return hashlib.sha256(b).hexdigest()

if os.path.exists(DEST):
    raise SystemExit(f"Ya existe {DEST}: no se sobrescribe. Si hay que registrar otra versión, avísame.")

crudo = open(CUAD, "rb").read()
nb = json.loads(crudo.decode("utf-8"))
codigo = [c for c in nb["cells"] if c["cell_type"] == "code"]
fuentes = ["".join(c["source"]) for c in codigo]

# (a) controles: tres celdas de código, en orden, y sin haberse ejecutado todavía
titulos = [f.splitlines()[1].strip("# ").strip() if len(f.splitlines()) > 1 else "" for f in fuentes]
esperado = ["CELDA 1", "CELDA 2", "CELDA 3"]
if len(fuentes) != 3 or not all(t.startswith(e) for t, e in zip(titulos, esperado)):
    raise SystemExit(f"El cuaderno 19 no tiene las tres celdas de H3 en orden: {titulos}. No se registra nada.")
otras = [c["cell_type"] for c in nb["cells"] if c["cell_type"] != "code"]
con_salidas = [i + 1 for i, c in enumerate(codigo) if c.get("outputs")]
if con_salidas:
    print(f"AVISO: las celdas {con_salidas} tienen salidas guardadas. La huella del código sigue valiendo, "
          "pero la del archivo cambia cada vez que se guarda.")

# (b) ¿ya llegaron las hojas? (se anota, para que conste que la huella es anterior)
hojas = sorted(os.listdir(LLENAS)) if os.path.isdir(LLENAS) else []

# (c) huellas: del archivo tal como está y del código de las tres celdas (la que importa)
SEP = "\n# ----- fin de celda -----\n"
registro = {
    "que_es": "Huella del cuaderno 19 (análisis de H3) registrada antes de recibir las hojas de los revisores",
    "fecha_hora": datetime.now().isoformat(timespec="seconds"),
    "archivo": CUAD,
    "bytes": len(crudo),
    "modificado": datetime.fromtimestamp(os.path.getmtime(CUAD)).isoformat(timespec="seconds"),
    "huella_archivo_sha256": sha_bytes(crudo),
    "huella_codigo_sha256": sha_bytes(SEP.join(fuentes).encode("utf-8")),
    "huellas_por_celda": {f"celda_{i + 1}": sha_bytes(f.encode("utf-8")) for i, f in enumerate(fuentes)},
    "titulos": titulos,
    "celdas_que_no_son_codigo": len(otras),
    "celdas_con_salidas": con_salidas,
    "carpeta_revision_h3_llenas": hojas if hojas else "no existe o está vacía",
    "como_verificar": ("Unir el código de las tres celdas de código con el separador "
                       + repr(SEP) + " y calcular SHA-256 en UTF-8; debe dar huella_codigo_sha256. "
                       "La huella del archivo cambia al guardar con salidas; la del código no."),
    "python": platform.python_version(),
}
with open(DEST, "x", encoding="utf-8") as f:          # "x": crea el archivo; si existe, falla
    json.dump(registro, f, ensure_ascii=False, indent=1)

print("Huella del cuaderno 19 registrada en", DEST)
print("  fecha y hora      :", registro["fecha_hora"])
print("  archivo           :", registro["bytes"], "bytes, modificado", registro["modificado"])
print("  huella del archivo:", registro["huella_archivo_sha256"])
print("  huella del código :", registro["huella_codigo_sha256"])
for k, v in registro["huellas_por_celda"].items():
    print(f"  {k}           : {v[:16]}  ({titulos[int(k[-1]) - 1][:60]})")
print("  hojas de revisión :", registro["carpeta_revision_h3_llenas"])
print("Mándate por correo la huella del código, como hiciste con la de la clave.")

In [ ]:
# =====================================================================================
# MANIFIESTO DEL 25 DE OCTUBRE (para la tutora). Solo lee; escribe un único archivo nuevo:
#   C:\SERCOP_FINAL\plan_A\documentacion\manifiesto_2026-10-25.csv  (si ya existe, se detiene)
#   No toca planA_manifiesto.csv del 8 de octubre. No incluye la clave de H3, puntajes.parquet,
#   hojas de revisión ni planA_validacion_sri. No imprime ningún RUC.
# =====================================================================================
import os, glob, json, hashlib, duckdb, pandas as pd
from datetime import datetime

SERCOP = r"C:\SERCOP_FINAL"
PLAN = os.path.join(SERCOP, "plan_A")
DEST = os.path.join(PLAN, "documentacion", "manifiesto_2026-10-25.csv")
if os.path.exists(DEST):
    raise SystemExit(f"Ya existe {DEST}: no se sobrescribe.")

EXCLUIR = ("planA_validacion_sri", "puntajes", "CLAVE_H3", "clave_h3", "revisor_", "consenso_h3")
ROL = {
    "planA_procesos_modelo.parquet": "Base de modelado: un proceso por fila (llave ocid)",
    "planA_indicadores.parquet": "Indicadores por proceso, versión v6_2 (llave ocid)",
    "planA_dim_proveedor.parquet": "Dimensión de proveedores (llave ruc)",
    "planA_dim_entidad.parquet": "Dimensión de entidades (llave ruc)",
    "planA_universo.parquet": "Universo completo de releases (auditoría)",
    "planA_catalogo.parquet": "Catálogo electrónico (solo descriptivo)",
    "planA_excluidos_solo_planeacion.parquet": "Excluidos: solo llegaron a planeación",
    "planA_excluidos_sin_monto.parquet": "Excluidos: sin monto",
    "planA_excluidos_sin_proveedor.parquet": "Excluidos: sin proveedor identificable",
    "preregistro_v1.md": "Preregistro firmado",
    "preregistro_firma.json": "Firma del preregistro (huellas citadas)",
    "huella_cuaderno19.json": "Huella del cuaderno 19 antes de recibir las hojas de H3",
    "reproduccion_indicadores.json": "Reproducción de los indicadores desde sus columnas de origen (celda R)",
    "reproduccion_percentiles_detalle.csv": "Detalle de la reproducción de percentiles (celda R)",
    "enmienda_2026-10-16.json": "Enmienda A",
    "enmienda_2026-10-16_tabla_por_anio.csv": "Tabla por año de la enmienda A",
    "enmienda_2026-10-16_b.json": "Enmienda B",
    "enmienda_2026-10-16_c.json": "Enmienda C",
}

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

con = duckdb.connect()
def q(p):
    return p.replace("\\", "/")

# (a) qué entra: los parquet de la carpeta plan_A (sin subcarpetas), las enmiendas y el preregistro
candidatos = (sorted(glob.glob(os.path.join(PLAN, "*.parquet")))
              + sorted(glob.glob(os.path.join(SERCOP, "enmiendas", "*")))
              + sorted(glob.glob(os.path.join(SERCOP, "preregistro", "*"))))
fuera = [os.path.relpath(p, SERCOP).replace("/", "\\") for p in candidatos if any(x in p for x in EXCLUIR)]
entran = [p for p in candidatos if os.path.isfile(p) and not any(x in p for x in EXCLUIR)]
print(f"Archivos en el manifiesto: {len(entran)} | dejados fuera a propósito: {fuera if fuera else 'ninguno'}")

filas = []
for p in entran:
    rel = os.path.relpath(p, SERCOP).replace("/", "\\")
    nombre = os.path.basename(p)
    n_f = n_c = None
    if nombre.endswith(".parquet"):
        n_f = con.execute(f"SELECT COUNT(*) FROM read_parquet('{q(p)}')").fetchone()[0]
        n_c = len(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{q(p)}')").fetchall())
    elif nombre.endswith(".csv"):
        t = pd.read_csv(p, sep=None, engine="python", encoding="utf-8-sig")
        n_f, n_c = len(t), t.shape[1]
    filas.append(dict(
        archivo=rel,
        grupo=rel.split("\\")[0],
        descripcion=ROL.get(nombre, ""),
        filas=n_f, columnas=n_c,
        bytes=os.path.getsize(p),
        modificado=datetime.fromtimestamp(os.path.getmtime(p)).strftime("%Y-%m-%d %H:%M:%S"),
        sha256=sha(p)))
M = pd.DataFrame(filas)
M[["filas", "columnas"]] = M[["filas", "columnas"]].astype("Int64")      # enteros; vacío si no aplica
assert not M["archivo"].str.contains("|".join(EXCLUIR)).any()

# (b) huellas contra el preregistro firmado
FIRMA = json.load(open(os.path.join(SERCOP, "preregistro", "preregistro_firma.json"), encoding="utf-8"))
citadas = {k.replace("/", "\\").lower(): v for k, v in FIRMA["huellas_citadas"].items()}
citadas[os.path.join("preregistro", FIRMA["documento"]).lower()] = FIRMA["huella_documento"]
M["huella_en_preregistro"] = M["archivo"].str.lower().map(citadas)
M["coincide_con_preregistro"] = [("" if pd.isna(c) else ("sí" if c == s else "NO")) for c, s in zip(M["huella_en_preregistro"], M["sha256"])]
print("\nHuellas comparadas con preregistro_firma.json:")
for r in M[M["huella_en_preregistro"].notna()].itertuples():
    print(f"  {r.coincide_con_preregistro:<3} {r.archivo:<52} {r.sha256[:16]}")
claves = ["plan_A\\planA_indicadores.parquet", "plan_A\\planA_procesos_modelo.parquet"]
ok_nucleo = all((M.loc[M["archivo"].str.lower() == c.lower(), "coincide_con_preregistro"] == "sí").any() for c in claves)
print("Indicadores y procesos:", "IGUALES a las huellas del preregistro." if ok_nucleo
      else "DISTINTOS de las huellas del preregistro. No mandes nada y avísame.")
reg_h2 = glob.glob(os.path.join(SERCOP, "modelos", "*_h2_h3", "registro_h2.json"))
if reg_h2:
    h_c = str(json.load(open(reg_h2[0], encoding="utf-8")).get("enmienda_c", ""))
    fila_c = M[M["archivo"].str.endswith("enmienda_2026-10-16_c.json")]
    if len(fila_c) and h_c:
        igual = fila_c["sha256"].iloc[0].startswith(h_c[:16])
        print(f"Enmienda C contra registro_h2.json: {'sí coincide' if igual else 'NO coincide'} ({h_c[:16]})")

# (c) cadena de 2.785.355 a 446.677 con los conteos de cada archivo
def n(nombre):
    s = M.loc[M["archivo"].str.endswith(nombre), "filas"]
    return int(s.iloc[0]) if len(s) else None
partes = [("planA_catalogo.parquet", "catálogo electrónico"),
          ("planA_excluidos_solo_planeacion.parquet", "solo planeación"),
          ("planA_excluidos_sin_monto.parquet", "sin monto"),
          ("planA_excluidos_sin_proveedor.parquet", "sin proveedor identificable")]
u, base = n("planA_universo.parquet"), n("planA_procesos_modelo.parquet")
print(f"\nCadena de filtros (conteo de filas de cada archivo):\n  Universo                        {u:>12,}")
resto = u
for archivo, etiqueta in partes:
    resto -= n(archivo)
    print(f"  menos {etiqueta:<26}{n(archivo):>12,}   quedan {resto:>10,}")
print(f"  = base de modelado              {base:>12,}   {'cuadra' if resto == base else 'NO CUADRA'}")

# partición: cada ocid del universo está en uno y solo uno de los cinco archivos
sub = [archivo for archivo, _ in partes] + ["planA_procesos_modelo.parquet"]
union = " UNION ALL ".join(f"SELECT ocid, '{a}' AS archivo FROM read_parquet('{q(os.path.join(PLAN, a))}')" for a in sub)
UNI = f"read_parquet('{q(os.path.join(PLAN, 'planA_universo.parquet'))}')"
tot, dist, nulos = con.execute(f"SELECT COUNT(*), COUNT(DISTINCT ocid), COUNT(*) - COUNT(ocid) FROM ({union})").fetchone()
falta = con.execute(f"SELECT COUNT(*) FROM {UNI} u ANTI JOIN ({union}) s USING (ocid)").fetchone()[0]
sobra = con.execute(f"SELECT COUNT(*) FROM ({union}) s ANTI JOIN {UNI} u USING (ocid)").fetchone()[0]
particion = (tot == dist == u) and nulos == 0 and falta == 0 and sobra == 0
print(f"  Partición por ocid: {tot:,} filas en los cinco archivos, {dist:,} ocid distintos, {nulos:,} ocid nulos;")
print(f"    repetidos entre archivos: {tot - dist:,} | del universo sin archivo: {falta:,} | "
      f"en archivos pero no en el universo: {sobra:,} -> {'partición exacta' if particion else 'NO es una partición exacta'}")

# monto por archivo, cuando el archivo trae monto_final (millones USD)
print("\n  Monto (monto_final, millones USD) por archivo:")
for a in ["planA_universo.parquet"] + sub:
    cols = [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{q(os.path.join(PLAN, a))}')").fetchall()]
    if "monto_final" in cols:
        m = con.execute(f"SELECT SUM(monto_final)/1e6 FROM read_parquet('{q(os.path.join(PLAN, a))}')").fetchone()[0]
        print(f"    {a:<44} {0 if m is None else m:>12,.1f}")
    else:
        print(f"    {a:<44} {'sin monto_final':>12}")

# (d) guardar (archivo nuevo, separador ';' para abrirlo en Excel en español)
with open(DEST, "x", encoding="utf-8-sig", newline="") as f:
    M.to_csv(f, index=False, sep=";")
print(f"\nManifiesto guardado: {DEST} ({len(M)} archivos, huella SHA-256 completa en la columna sha256)")
print(M.assign(MB=(M["bytes"] / 1e6).round(1), sha256_16=M["sha256"].str[:16])
      [["archivo", "filas", "columnas", "MB", "modificado", "sha256_16"]].to_string(index=False))


In [ ]:
# =====================================================================================
# PUNTO 5 · RECONSTRUCCIÓN DE LA CONCURRENCIA Y DE LAS COLUMNAS _1 (solo lectura)
#   Rehace dif_concurrencia_grupo y ref_concurrencia_grupo con la regla de la celda 46 de
#   pruebas.ipynb (la que escribió la v6_2) y las compara con planA_indicadores valor por valor,
#   contando nulo contra nulo. Hace lo mismo con los cinco percentiles y sus copias _1.
#   No escribe ningún archivo. No imprime ningún RUC.
# =====================================================================================
import duckdb, pandas as pd

PLAN = r"C:\SERCOP_FINAL\plan_A"
IND = (PLAN + r"\planA_indicadores.parquet").replace("\\", "/")
PM = (PLAN + r"\planA_procesos_modelo.parquet").replace("\\", "/")
con = duckdb.connect()
con.execute(f"CREATE VIEW ind AS SELECT * FROM read_parquet('{IND}')")
con.execute(f"CREATE VIEW pm AS SELECT * FROM read_parquet('{PM}')")
N = con.execute("SELECT count(*) FROM ind").fetchone()[0]
cols_pm = [r[0] for r in con.execute("DESCRIBE SELECT * FROM pm").fetchall()]
print(f"planA_indicadores: {N:,} filas | planA_procesos_modelo: {con.execute('SELECT count(*) FROM pm').fetchone()[0]:,} filas")

# grupo comparable de la celda 46: familia|periodo si tiene 30 o más casos en el dominio, si no la
# familia, si no GLOBAL; los conteos se hacen dentro del dominio de cada variable
def dominio(tabla, x, filtro=""):
    return f"""
      SELECT ocid, x, CASE WHEN n_fp >= 30 THEN familia || '|' || periodo
                           WHEN n_f  >= 30 THEN familia ELSE 'GLOBAL' END AS grupo
      FROM (SELECT ocid, familia, periodo, {x} AS x,
                   count(*) OVER (PARTITION BY familia, periodo) AS n_fp,
                   count(*) OVER (PARTITION BY familia)          AS n_f
            FROM {tabla} WHERE {x} IS NOT NULL {filtro})"""

def comparar(nombre, sql_archivo, sql_recon):
    """Cuenta, sobre todas las filas, nulo contra nulo y valor contra valor."""
    r = con.execute(f"""
      WITH a AS ({sql_archivo}), b AS ({sql_recon})
      SELECT count(*),
             count(*) FILTER (WHERE a.v IS NULL AND b.v IS NULL),
             count(*) FILTER (WHERE a.v IS NOT NULL AND b.v IS NOT NULL AND abs(a.v - b.v) <= 1e-9),
             count(*) FILTER (WHERE a.v IS NOT NULL AND b.v IS NOT NULL AND abs(a.v - b.v) > 1e-9),
             count(*) FILTER (WHERE a.v IS NULL AND b.v IS NOT NULL),
             count(*) FILTER (WHERE a.v IS NOT NULL AND b.v IS NULL),
             max(abs(a.v - b.v))
      FROM a LEFT JOIN b USING (ocid)""").fetchone()
    tot, nn, igual, dist, solo_rec, solo_arch, maxd = r
    ok = (nn + igual == tot)
    return dict(columna=nombre, filas=tot, ambos_nulos=nn, iguales=igual, distintos=dist,
                nulo_en_archivo=solo_rec, nulo_en_reconstruccion=solo_arch,
                max_dif=0.0 if maxd is None else round(float(maxd), 12),
                veredicto="IDÉNTICA" if ok else "DIFIERE")

# (a) la fuente: ind_n_oferentes del archivo de indicadores contra n_oferentes de la base
print("\n(a) Fuente del número de oferentes")
fuente_pm = "n_oferentes" if "n_oferentes" in cols_pm else None
if fuente_pm:
    print(pd.DataFrame([comparar("ind_n_oferentes vs base.n_oferentes",
        "SELECT ocid, ind_n_oferentes::DOUBLE AS v FROM ind",
        f"SELECT ocid, {fuente_pm}::DOUBLE AS v FROM pm")]).to_string(index=False))
else:
    print("  La base no tiene una columna n_oferentes; la reconstrucción usa ind_n_oferentes.")
    print("  Columnas de la base con 'ofer' en el nombre:", [c for c in cols_pm if "ofer" in c])

# (b) concurrencia con la regla de la celda 46: dominio = dos o más oferentes; referencia = mediana
#     del grupo; dif = referencia - oferentes (más alto = menos competencia que el grupo)
print("\n(b) Concurrencia reconstruida con la regla de la celda 46 (dominio: dos o más oferentes)")
filas = []
for etiqueta, tabla in [("desde los indicadores", "ind"),
                        ("desde la base", "(SELECT i.ocid, i.familia, i.periodo, p.n_oferentes AS ind_n_oferentes "
                                          "FROM ind i JOIN pm p USING (ocid))" if fuente_pm else None)]:
    if tabla is None:
        continue
    e = dominio(tabla, "ind_n_oferentes", "AND ind_n_oferentes > 1")
    rec = f"""WITH e AS ({e}), m AS (SELECT grupo, median(x) AS med FROM e GROUP BY grupo)
              SELECT e.ocid, (m.med - e.x)::DOUBLE AS dif, m.med::DOUBLE AS ref FROM e JOIN m USING (grupo)"""
    for col, campo in [("dif_concurrencia_grupo", "dif"), ("ref_concurrencia_grupo", "ref")]:
        f = comparar(col, f"SELECT ocid, {col}::DOUBLE AS v FROM ind",
                     f"SELECT ocid, {campo} AS v FROM ({rec})")
        f["reconstruida"] = etiqueta
        filas.append(f)
print(pd.DataFrame(filas)[["reconstruida", "columna", "filas", "ambos_nulos", "iguales", "distintos",
                           "nulo_en_archivo", "nulo_en_reconstruccion", "max_dif", "veredicto"]].to_string(index=False))
dom = con.execute("SELECT count(*) FILTER (WHERE ind_n_oferentes > 1), count(*) FILTER (WHERE ind_n_oferentes <= 1),"
                  " count(*) FILTER (WHERE ind_n_oferentes IS NULL) FROM ind").fetchone()
print(f"  Dominio: {dom[0]:,} procesos con dos o más oferentes; fuera: {dom[1]:,} con uno o ninguno y "
      f"{dom[2]:,} sin dato (suman {dom[1] + dom[2]:,}, los nulos esperados)")
print("  Referencia por grupo (mediana de oferentes entre los procesos con dos o más):")
print(con.execute(f"""WITH e AS ({dominio('ind', 'ind_n_oferentes', 'AND ind_n_oferentes > 1')})
    SELECT grupo, count(*) AS procesos, median(x) AS mediana_oferentes FROM e GROUP BY 1 ORDER BY 1""").df().to_string(index=False))

# (c) los cinco percentiles y sus copias _1: ¿cuál es mitad de salto (v6.2), cuál cume_dist (v6.1)
#     y cuál percent_rank (v6, celda 44)?
print("\n(c) Percentiles y columnas _1 contra las tres fórmulas (mismo grupo y mismo orden de la celda 46)")
CFG = [("pct_precio_alto", "ind_desv_precio_unit_z", "ASC"),
       ("pct_monto_capital", "ind_monto_sobre_capital", "ASC"),
       ("pct_proveedor_nuevo", "ind_antiguedad_dias", "DESC"),
       ("pct_plazo_corto", "ind_plazo_ofertas_dias", "DESC"),
       ("pct_firma_rapida", "ind_plazo_adj_firma_dias", "DESC")]
cols_ind = [r[0] for r in con.execute("DESCRIBE SELECT * FROM ind").fetchall()]
res = []
for pct, src, orden in CFG:
    e = dominio("ind", f'"{src}"')
    formulas = {
        "mitad de salto (v6.2)": f"""cume_dist() OVER (PARTITION BY grupo ORDER BY x {orden})
              - (count(*) OVER (PARTITION BY grupo, x))::DOUBLE / (2.0 * count(*) OVER (PARTITION BY grupo))""",
        "cume_dist (v6.1)": f"cume_dist() OVER (PARTITION BY grupo ORDER BY x {orden})",
        "percent_rank (v6)": f"percent_rank() OVER (PARTITION BY grupo ORDER BY x {orden})"}
    for col in [pct, pct + "_1"]:
        if col not in cols_ind:
            continue
        fila = {"columna": col}
        for nombre_f, expr in formulas.items():
            c = comparar(col, f'SELECT ocid, "{col}"::DOUBLE AS v FROM ind',
                         f"SELECT ocid, ({expr})::DOUBLE AS v FROM ({e})")
            fila[nombre_f] = c["veredicto"] if c["veredicto"] == "IDÉNTICA" else f"no ({c['distintos']:,} distintos)"
        m = con.execute(f"""WITH e AS ({e}), a AS (SELECT e.grupo, i."{col}" AS v FROM e JOIN ind i USING (ocid))
                            SELECT max(abs(g - 0.5)) FROM (SELECT grupo, avg(v) AS g FROM a GROUP BY grupo)""").fetchone()[0]
        fila["max |media del grupo - 0,5|"] = round(float(m), 6)
        res.append(fila)
print(pd.DataFrame(res).to_string(index=False))
print(f"\nLectura: 'IDÉNTICA' quiere decir que las {N:,} filas coinciden (nulo con nulo y valor con valor,"
      "\ntolerancia 1e-9). La regla de conservación de la v6.2 pide media 0,5000 dentro de cada grupo.")

## C. Verificación de la huella del cuaderno 19

Correr **antes de la celda 3 de H3** con las hojas reales (y cuando se quiera). Compara el código actual de las tres celdas de `plan_A\cuaderno19.ipynb` con la huella registrada el 25 de octubre a las 19:08:27. Si dice DISTINTO, no se corre la celda 3.

In [ ]:
# =====================================================================================
# VERIFICAR QUE EL CÓDIGO DEL CUADERNO 19 NO CAMBIÓ (correr cuando quieras, cuantas veces quieras)
#   Solo lee. Compara el código actual de las tres celdas con la huella registrada.
# =====================================================================================
import os, json, hashlib
SERCOP = r"C:\SERCOP_FINAL"
REG = json.load(open(os.path.join(SERCOP, "preregistro", "huella_cuaderno19.json"), encoding="utf-8"))
nb = json.load(open(os.path.join(SERCOP, "plan_A", "cuaderno19.ipynb"), encoding="utf-8"))
fuentes = ["".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code"]
SEP = "\n# ----- fin de celda -----\n"
actual = hashlib.sha256(SEP.join(fuentes).encode("utf-8")).hexdigest()
print("Registrada el", REG["fecha_hora"], "|", REG["huella_codigo_sha256"][:16])
print("Código actual            |", actual[:16], "|", len(fuentes), "celdas de código")
if actual == REG["huella_codigo_sha256"]:
    print("IGUAL: el código del cuaderno 19 es el mismo que se registró.")
else:
    for i, f in enumerate(fuentes, start=1):
        h = hashlib.sha256(f.encode("utf-8")).hexdigest()
        print(f"  celda {i}:", "igual" if h == REG["huellas_por_celda"].get(f"celda_{i}") else "DISTINTA")
    print("DISTINTO: el código cambió después de registrar la huella. No sigas con H3 y avísame.")

## D. Inventario y uniones del 20 de octubre (solo lectura)

In [ ]:
# ==========================================================
# CELDA: DÓNDE ESTÁN LOS DATOS DEL PLAN A (solo lectura)
# No crea, modifica ni borra nada. No abre planA_validacion_sri,
# no imprime RUC y no abre puntajes.parquet.
# ==========================================================
from pathlib import Path
import duckdb

RAIZ = Path(r"C:\SERCOP_FINAL")
PLAN_A = RAIZ / "plan_A"


def miles(n):
    """Escribe 446677 como 446.677."""
    return f"{n:,}".replace(",", ".")


# --- 1. Carpetas principales ---
print("Carpeta de trabajo:", RAIZ, "| existe:", RAIZ.exists())
print("Datos del Plan A  :", PLAN_A, "| existe:", PLAN_A.exists())

# --- 2. Qué hay dentro de plan_A (solo nombre y tamaño, sin abrir) ---
print("\nContenido de plan_A (sin abrir los archivos):")
if PLAN_A.exists():
    for f in sorted(PLAN_A.iterdir()):
        if f.is_file():
            print(f"  {f.name:<42}{f.stat().st_size / 1e6:>10.1f} MB")
        else:
            print(f"  {f.name:<42}{'(carpeta)':>13}")

# --- 3. Archivos que usan los modelos: filas y columnas ---
# Universo de cada cifra: total de filas del archivo, sin ningún filtro.
ESPERADO = {
    "planA_procesos_modelo.parquet": 446677,
    "planA_indicadores.parquet": None,  # no consta en los documentos
    "planA_dim_proveedor.parquet": 74778,
    "planA_dim_entidad.parquet": 4244,
    "planA_universo.parquet": 2785355,
    "planA_catalogo.parquet": 2285455,
}

con = duckdb.connect()  # base en memoria: no crea ningún archivo
columnas = {}

print("\nFilas y columnas de cada archivo (filas = todo el archivo, sin filtros):")
print(f"{'archivo':<34}{'filas':>12}{'columnas':>10}  contra los documentos")
for nombre, esperado in ESPERADO.items():
    ruta = PLAN_A / nombre
    if not ruta.exists():
        print(f"{nombre:<34}  NO ENCONTRADO (revisar el nombre en la lista de arriba)")
        continue
    p = ruta.as_posix()
    filas = con.execute(f"SELECT COUNT(*) FROM read_parquet('{p}')").fetchone()[0]
    describe = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{p}')").fetchall()
    columnas[nombre] = [fila[0] for fila in describe]
    if esperado is None:
        estado = "sin cifra de referencia"
    elif filas == esperado:
        estado = "coincide"
    else:
        estado = f"DIFIERE (documentos: {miles(esperado)})"
    print(f"{nombre:<34}{miles(filas):>12}{len(columnas[nombre]):>10}  {estado}")

# --- 4. Columnas de unión (solo los nombres, nunca los valores) ---
print("\nColumnas de unión:")
PROC = columnas.get("planA_procesos_modelo.parquet", [])
IND = columnas.get("planA_indicadores.parquet", [])
for clave in ["ocid", "proveedor_ruc_final", "entidad_ruc"]:
    print(f"  {clave:<22} en procesos_modelo: {'sí' if clave in PROC else 'FALTA'}")
print(f"  {'ocid':<22} en indicadores:     {'sí' if 'ocid' in IND else 'FALTA'}")
for nombre in ["planA_dim_proveedor.parquet", "planA_dim_entidad.parquet"]:
    con_ruc = [c for c in columnas.get(nombre, []) if "ruc" in c.lower()]
    print(f"  columnas con 'ruc' en {nombre}: {con_ruc}")

con.close()

# --- 5. Corridas de modelos y hojas de los revisores (solo nombres) ---
MODELOS = RAIZ / "modelos"
print("\nCorridas guardadas en", MODELOS)
if MODELOS.exists():
    for c in sorted(MODELOS.iterdir()):
        if c.is_dir():
            print("  ", c.name)
else:
    print("   la carpeta no existe")

REVISION = RAIZ / "revision_h3_llenas"
print("\nHojas de los revisores en", REVISION)
if REVISION.exists():
    for f in sorted(REVISION.iterdir()):
        print("  ", f.name)
else:
    print("   la carpeta no existe todavía")

# --- 6. Cuadernos .ipynb que hay en la carpeta de trabajo ---
print("\nCuadernos .ipynb en", RAIZ)
for f in sorted(RAIZ.rglob("*.ipynb")):
    if ".ipynb_checkpoints" not in f.parts:
        print("  ", f.relative_to(RAIZ))

In [ ]:
# ==========================================================
# CELDA: CUADERNOS, FECHAS Y COLUMNAS (solo lectura)
# De cada cuaderno lee solo el título (primera celda de texto).
# No lee las salidas de las celdas, no abre planA_validacion_sri ni
# puntajes.parquet y no imprime RUC ni valores de datos.
# ==========================================================
import json
import textwrap
from datetime import datetime
from pathlib import Path
import duckdb

RAIZ = Path(r"C:\SERCOP_FINAL")
PLAN_A = RAIZ / "plan_A"


def fecha(ruta):
    """Fecha y hora de la última modificación del archivo."""
    return datetime.fromtimestamp(ruta.stat().st_mtime).strftime("%Y-%m-%d %H:%M")


def titulo_cuaderno(ruta):
    """Primera línea de la primera celda de texto. No toca las salidas."""
    with open(ruta, "r", encoding="utf-8") as f:
        nb = json.load(f)
    celdas = nb.get("cells", [])
    n_codigo = sum(1 for c in celdas if c.get("cell_type") == "code")
    for c in celdas:
        if c.get("cell_type") == "markdown":
            fuente = c.get("source", "")
            if isinstance(fuente, list):
                fuente = "".join(fuente)
            for linea in fuente.splitlines():
                if linea.strip():
                    return linea.strip()[:90], n_codigo
    return "(sin celda de texto)", n_codigo


# --- 1. Cuadernos: tamaño, fecha, celdas de código y título ---
print("1) CUADERNOS (.ipynb)")
for f in sorted(RAIZ.rglob("*.ipynb")):
    if ".ipynb_checkpoints" in f.parts:
        continue
    rel = f.relative_to(RAIZ)
    try:
        titulo, n_cod = titulo_cuaderno(f)
    except Exception as e:
        titulo, n_cod = f"(no se pudo leer: {type(e).__name__})", "?"
    print(f"\n  {rel}")
    print(f"     {f.stat().st_size / 1e3:,.0f} KB | modificado {fecha(f)} | celdas de código: {n_cod}")
    print(f"     título: {titulo}")

# --- 2. Fecha de modificación de los archivos de datos de plan_A ---
print("\n2) FECHA DE LOS ARCHIVOS DE DATOS (plan_A)")
for nombre in ["planA_procesos_modelo.parquet", "planA_indicadores.parquet",
               "planA_dim_proveedor.parquet", "planA_dim_entidad.parquet",
               "planA_universo.parquet", "planA_catalogo.parquet"]:
    ruta = PLAN_A / nombre
    if ruta.exists():
        print(f"  {nombre:<34} modificado {fecha(ruta)}")
    else:
        print(f"  {nombre:<34} NO ENCONTRADO")

# --- 3. Nombres de las columnas de planA_indicadores (solo nombres) ---
print("\n3) COLUMNAS DE planA_indicadores.parquet (solo nombres)")
ruta_ind = PLAN_A / "planA_indicadores.parquet"
if ruta_ind.exists():
    con = duckdb.connect()
    desc = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{ruta_ind.as_posix()}')").fetchall()
    con.close()
    nombres = [d[0] for d in desc]
    print(f"  Total: {len(nombres)} columnas")
    print(textwrap.fill(", ".join(nombres), width=100,
                        initial_indent="  ", subsequent_indent="  "))
else:
    print("  NO ENCONTRADO")

# --- 4. Carpetas de primer nivel y contenido de dos corridas (solo nombres) ---
print("\n4) CARPETAS DE PRIMER NIVEL EN", RAIZ)
for f in sorted(RAIZ.iterdir()):
    if f.is_dir():
        print("  ", f.name)

for corrida in ["20261016_1845_h2_h3", "20261016_2000_h3"]:
    carpeta = RAIZ / "modelos" / corrida
    print(f"\n5) ARCHIVOS DE modelos\\{corrida} (solo nombres)")
    if carpeta.exists():
        for f in sorted(carpeta.iterdir()):
            if f.is_file():
                print(f"   {f.name:<45}{f.stat().st_size / 1e3:>10,.0f} KB  {fecha(f)}")
            else:
                print(f"   {f.name:<45}{'(carpeta)':>13}")
    else:
        print("   la carpeta no existe")

In [ ]:
# ==========================================================
# CELDA: ARCHIVOS DE SALIDA Y SU FECHA (solo lectura)
# Sirve para emparejar cada cuaderno con lo que produjo, por la fecha.
# Solo muestra ruta, tamaño y fecha. No abre ningún archivo.
# ==========================================================
import os
from datetime import datetime
from fnmatch import fnmatch
from pathlib import Path

RAIZ = Path(r"C:\SERCOP_FINAL")

# Carpetas que no se recorren (JSON crudos y temporales)
OMITIR = {"00_json_crudo", ".tmp", "tmp_duckdb"}

PATRONES = [
    "c9_*",
    "c14_verificacion_releases.csv",
    "c8_especificacion_congelada.json",
    "especificacion_indicadores.json",
    "final_procesos.parquet",
    "final_panel.parquet",
    "planA_manifiesto.csv",
    "enmienda_*.json",
    "reproduccion_indicadores.json",
    "preregistro_firma.json",
]

encontrados = []
for carpeta, subcarpetas, archivos in os.walk(RAIZ):
    subcarpetas[:] = [d for d in subcarpetas if d not in OMITIR]
    for nombre in archivos:
        if any(fnmatch(nombre, p) for p in PATRONES):
            ruta = Path(carpeta) / nombre
            estado = ruta.stat()
            encontrados.append((str(ruta.relative_to(RAIZ)), estado.st_size / 1e3, estado.st_mtime))

print(f"Archivos encontrados: {len(encontrados)}\n")
for rel, kb, mtime in sorted(encontrados):
    fecha = datetime.fromtimestamp(mtime).strftime("%Y-%m-%d %H:%M")
    print(f"{rel:<75}{kb:>10,.0f} KB  {fecha}")

In [ ]:
# ==========================================================
# CELDA: UNIONES, VERSIONES DE INDICADORES Y PAQUETE DE COMPAÑEROS
# Solo lectura. Imprime conteos, huellas y nombres de columnas.
# No imprime ningún RUC, no abre planA_validacion_sri ni puntajes.parquet.
# Universo de cada cifra: todas las filas del archivo indicado, sin filtros.
# ==========================================================
import hashlib
import json
import re
import textwrap
from pathlib import Path
import duckdb

RAIZ = Path(r"C:\SERCOP_FINAL")
PA = RAIZ / "plan_A"

# Archivos principales (los cuatro del núcleo de modelado)
P = PA / "planA_procesos_modelo.parquet"
I = PA / "planA_indicadores.parquet"
DP = PA / "planA_dim_proveedor.parquet"
DE = PA / "planA_dim_entidad.parquet"

con = duckdb.connect()  # base en memoria: no crea archivos


def q(ruta):
    """Ruta en el formato que entiende DuckDB."""
    return Path(ruta).as_posix()


def miles(n):
    return "sin dato" if n is None else f"{n:,}".replace(",", ".")


def uno(sql):
    return con.execute(sql).fetchone()


def columnas(ruta):
    filas = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{q(ruta)}')").fetchall()
    return [f[0] for f in filas]


def tipos(ruta):
    filas = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{q(ruta)}')").fetchall()
    return {f[0]: f[1] for f in filas}


def huella(ruta):
    """Primeros 16 caracteres del SHA-256 del archivo."""
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()[:16]


def seguro(titulo, funcion):
    """Si una parte falla, lo dice y sigue con la siguiente."""
    print(f"\n{titulo}")
    try:
        funcion()
    except Exception as e:
        msg = re.sub(r"\d{10,}", "[NUM]", str(e))[:160]
        print(f"   NO SE PUDO: {type(e).__name__}: {msg}")


# ---------- 1. Huellas: qué archivo es copia de cuál ----------
def parte1():
    grupos = {
        "planA_indicadores": [
            ("actual", I),
            ("_versiones\\v6_2", PA / "_versiones" / "planA_indicadores_v6_2.parquet"),
            ("_versiones\\v6_1", PA / "_versiones" / "planA_indicadores_v6_1.parquet"),
            ("_envio_companeros", PA / "_envio_companeros" / "planA_indicadores.parquet"),
            ("_version_1_historico\\v4_3", PA / "_version_1_historico" / "planA_indicadores_v4_3.parquet"),
        ],
        "planA_dim_entidad": [
            ("actual", DE),
            ("_envio_companeros", PA / "_envio_companeros" / "planA_dim_entidad.parquet"),
            ("_versiones\\previo_tipologia", PA / "_versiones" / "planA_dim_entidad_previo_tipologia.parquet"),
        ],
        "planA_procesos_modelo": [
            ("actual", P),
            ("_envio_companeros", PA / "_envio_companeros" / "planA_procesos_modelo.parquet"),
        ],
        "planA_dim_proveedor": [
            ("actual", DP),
            ("_envio_companeros", PA / "_envio_companeros" / "planA_dim_proveedor.parquet"),
        ],
    }
    actuales = {}
    for nombre, lista in grupos.items():
        print(f"  {nombre}")
        ref = None
        for etiqueta, ruta in lista:
            if not ruta.exists():
                print(f"     {etiqueta:<30} NO EXISTE")
                continue
            h = huella(ruta)
            if ref is None:
                ref = h
                actuales[nombre] = h
                marca = "(referencia)"
            else:
                marca = "IGUAL al actual" if h == ref else "distinto del actual"
            print(f"     {etiqueta:<30}{ruta.stat().st_size / 1e6:>8.1f} MB  {h}  {marca}")
    # huellas que citan los documentos
    print("  Huellas citadas para planA_indicadores:")
    espec = PA / "documentacion" / "especificacion_indicadores.json"
    if espec.exists():
        d = json.loads(espec.read_text(encoding="utf-8"))
        print("     especificación, campo sha256_del_parquet_indicadores :", str(d.get("sha256_del_parquet_indicadores", ""))[:16])
        enm = d.get("enmiendas", [])
        if enm:
            print("     especificación, última enmienda (v6)                :", enm[-1].get("sha256_16_indicadores_v6"))
    print("     documentos, primera corrida de modelos (11-oct)     : dc87f4b8e1282d7f")
    print("     huella del archivo actual                           :", actuales.get("planA_indicadores"))


seguro("1) HUELLAS (SHA-256, 16 caracteres)", parte1)


# ---------- 2. Columnas entre versiones de indicadores ----------
def comparar(nom_a, ruta_a, nom_b, ruta_b):
    if not (ruta_a.exists() and ruta_b.exists()):
        print(f"  {nom_a} contra {nom_b}: falta un archivo")
        return
    ca, cb = columnas(ruta_a), columnas(ruta_b)
    solo_a = [c for c in ca if c not in cb]
    solo_b = [c for c in cb if c not in ca]
    print(f"  {nom_a} ({len(ca)} columnas) contra {nom_b} ({len(cb)} columnas)")
    for texto, lista in ((f"solo en {nom_a}", solo_a), (f"solo en {nom_b}", solo_b)):
        cuerpo = ", ".join(lista) if lista else "ninguna"
        print(textwrap.fill(f"{texto} ({len(lista)}): {cuerpo}", width=100,
                            initial_indent="     ", subsequent_indent="        "))


def parte2():
    comparar("v4_3 (paquete de compañeros)", PA / "_version_1_historico" / "planA_indicadores_v4_3.parquet",
             "actual", I)
    comparar("v6_1", PA / "_versiones" / "planA_indicadores_v6_1.parquet",
             "v6_2", PA / "_versiones" / "planA_indicadores_v6_2.parquet")
    comparar("dim_entidad previa", PA / "_versiones" / "planA_dim_entidad_previo_tipologia.parquet",
             "dim_entidad actual", DE)


seguro("2) COLUMNAS QUE CAMBIARON ENTRE VERSIONES (solo nombres)", parte2)


# ---------- 3. Las columnas con sufijo _1 y las de concurrencia ----------
def parte3():
    ci = columnas(I)
    bases = ["pct_precio_alto", "pct_monto_capital", "pct_proveedor_nuevo", "pct_plazo_corto", "pct_firma_rapida"]
    print(f"  {'columna':<22}{'nulos sin _1':>14}{'nulos con _1':>14}{'iguales':>10}{'dif. media':>12}{'correlación':>13}")
    for b in bases:
        b1 = b + "_1"
        if b in ci and b1 in ci:
            r = uno(f"""SELECT count(*) FILTER (WHERE {b} IS NULL), count(*) FILTER (WHERE {b1} IS NULL),
                               count(*) FILTER (WHERE {b} IS NOT DISTINCT FROM {b1}),
                               avg(abs({b} - {b1})), corr({b}, {b1}), count(*)
                        FROM read_parquet('{q(I)}')""")
            dif = "sin dato" if r[3] is None else f"{r[3]:.4f}"
            cor = "sin dato" if r[4] is None else f"{r[4]:.4f}"
            print(f"  {b:<22}{miles(r[0]):>14}{miles(r[1]):>14}{miles(r[2]):>10}{dif:>12}{cor:>13}")
        else:
            print(f"  {b:<22} falta alguna de las dos columnas")
    print("  (iguales = filas donde las dos columnas valen lo mismo, contando nulo contra nulo)")
    print("  Columnas de concurrencia (universo: todas las filas de planA_indicadores):")
    for c in ["pct_concurrencia_pos", "dif_concurrencia_grupo", "ref_concurrencia_grupo", "ind_n_oferentes"]:
        if c in ci:
            r = uno(f"SELECT count(*) FILTER (WHERE {c} IS NULL), min({c}), max({c}), count(DISTINCT {c}) "
                    f"FROM read_parquet('{q(I)}')")
            print(f"     {c:<26} nulos {miles(r[0]):>9} | mínimo {r[1]} | máximo {r[2]} | valores distintos {miles(r[3])}")
        else:
            print(f"     {c:<26} NO EXISTE en el archivo")


seguro("3) COLUMNAS CON SUFIJO _1 Y DE CONCURRENCIA", parte3)


# ---------- 4. Uniones: tipos, llaves únicas y cobertura ----------
def parte4():
    print("  Tipos de las columnas de unión:")
    for nombre, ruta, cols in [("procesos_modelo", P, ["ocid", "proveedor_ruc_final", "entidad_ruc", "monto_final"]),
                               ("indicadores", I, ["ocid", "proveedor_ruc", "entidad_ruc", "monto_final"]),
                               ("dim_proveedor", DP, ["ruc"]),
                               ("dim_entidad", DE, ["ruc", "entidad_ruc"])]:
        t = tipos(ruta)
        print("     " + nombre + ": " + ", ".join(f"{c} {t.get(c, 'NO EXISTE')}" for c in cols))

    print("\n  Llaves únicas (filas | valores distintos | nulos):")
    for nombre, ruta, col in [("procesos_modelo", P, "ocid"), ("indicadores", I, "ocid"),
                              ("dim_proveedor", DP, "ruc"), ("dim_entidad", DE, "ruc"),
                              ("dim_entidad", DE, "entidad_ruc")]:
        r = uno(f"SELECT count(*), count(DISTINCT {col}), count(*) FILTER (WHERE {col} IS NULL) "
                f"FROM read_parquet('{q(ruta)}')")
        print(f"     {nombre + '.' + col:<30}{miles(r[0]):>10} |{miles(r[1]):>10} |{miles(r[2]):>8}")

    print("\n  Procesos que están en un archivo y no en el otro (por ocid):")
    a = uno(f"SELECT count(*) FROM read_parquet('{q(P)}') p WHERE NOT EXISTS "
            f"(SELECT 1 FROM read_parquet('{q(I)}') i WHERE i.ocid = p.ocid)")[0]
    b = uno(f"SELECT count(*) FROM read_parquet('{q(I)}') i WHERE NOT EXISTS "
            f"(SELECT 1 FROM read_parquet('{q(P)}') p WHERE p.ocid = i.ocid)")[0]
    print(f"     en procesos_modelo y no en indicadores: {miles(a)} | en indicadores y no en procesos_modelo: {miles(b)}")

    print("\n  Cobertura de proveedor (procesos_modelo.proveedor_ruc_final contra dim_proveedor.ruc):")
    r = uno(f"""SELECT count(*), count(*) FILTER (WHERE p.proveedor_ruc_final IS NULL),
                       count(*) FILTER (WHERE p.proveedor_ruc_final IS NOT NULL AND d.ruc IS NULL)
                FROM read_parquet('{q(P)}') p
                LEFT JOIN (SELECT DISTINCT ruc FROM read_parquet('{q(DP)}')) d ON p.proveedor_ruc_final = d.ruc""")
    print(f"     procesos: {miles(r[0])} | sin RUC de proveedor: {miles(r[1])} | con RUC que no está en la dimensión: {miles(r[2])}")
    r = uno(f"""SELECT count(*), count(*) FILTER (WHERE d.ruc IS NOT NULL)
                FROM (SELECT DISTINCT proveedor_ruc_final AS r FROM read_parquet('{q(P)}')
                      WHERE proveedor_ruc_final IS NOT NULL) x
                LEFT JOIN (SELECT DISTINCT ruc FROM read_parquet('{q(DP)}')) d ON x.r = d.ruc""")
    print(f"     proveedores distintos en procesos: {miles(r[0])} | encontrados en la dimensión: {miles(r[1])}")
    r = uno(f"""SELECT count(*) FROM (SELECT DISTINCT ruc FROM read_parquet('{q(DP)}')) d
                WHERE NOT EXISTS (SELECT 1 FROM read_parquet('{q(P)}') p WHERE p.proveedor_ruc_final = d.ruc)""")
    print(f"     proveedores de la dimensión que no aparecen en procesos: {miles(r[0])}")

    for col in ["ruc", "entidad_ruc"]:
        print(f"\n  Cobertura de entidad (procesos_modelo.entidad_ruc contra dim_entidad.{col}):")
        r = uno(f"""SELECT count(*), count(*) FILTER (WHERE p.entidad_ruc IS NULL),
                           count(*) FILTER (WHERE p.entidad_ruc IS NOT NULL AND d.k IS NULL)
                    FROM read_parquet('{q(P)}') p
                    LEFT JOIN (SELECT DISTINCT {col} AS k FROM read_parquet('{q(DE)}')) d ON p.entidad_ruc = d.k""")
        print(f"     procesos: {miles(r[0])} | sin RUC de entidad: {miles(r[1])} | con RUC que no está en la dimensión: {miles(r[2])}")
        r = uno(f"""SELECT count(*), count(*) FILTER (WHERE d.k IS NOT NULL)
                    FROM (SELECT DISTINCT entidad_ruc AS r FROM read_parquet('{q(P)}')
                          WHERE entidad_ruc IS NOT NULL) x
                    LEFT JOIN (SELECT DISTINCT {col} AS k FROM read_parquet('{q(DE)}')) d ON x.r = d.k""")
        print(f"     entidades distintas en procesos: {miles(r[0])} | encontradas en la dimensión: {miles(r[1])}")

    print("\n  Las uniones no deben cambiar el número de filas:")
    base = uno(f"SELECT count(*) FROM read_parquet('{q(P)}')")[0]
    for col in ["ruc", "entidad_ruc"]:
        r = uno(f"""SELECT count(*) FROM read_parquet('{q(P)}') p
                    LEFT JOIN read_parquet('{q(DP)}') d ON p.proveedor_ruc_final = d.ruc
                    LEFT JOIN read_parquet('{q(DE)}') e ON p.entidad_ruc = e.{col}""")[0]
        estado = "igual, bien" if r == base else "CAMBIA: la llave se repite en una dimensión"
        print(f"     procesos {miles(base)} -> con ambas dimensiones (entidad por {col}) {miles(r)}: {estado}")

    print("\n  Procesos contra indicadores, emparejados por ocid (nulo contra nulo):")
    r = uno(f"""SELECT count(*),
                       count(*) FILTER (WHERE p.proveedor_ruc_final IS DISTINCT FROM i.proveedor_ruc),
                       count(*) FILTER (WHERE p.entidad_ruc IS DISTINCT FROM i.entidad_ruc),
                       count(*) FILTER (WHERE abs(coalesce(p.monto_final, -1) - coalesce(i.monto_final, -1)) > 0.005)
                FROM read_parquet('{q(P)}') p JOIN read_parquet('{q(I)}') i ON p.ocid = i.ocid""")
    print(f"     emparejados: {miles(r[0])} | proveedor distinto: {miles(r[1])} | entidad distinta: {miles(r[2])} | monto_final distinto: {miles(r[3])}")

    print("\n  Largo de los RUC (cuántos caracteres tienen; solo el conteo, nunca el valor):")
    for nombre, ruta, col in [("procesos_modelo.proveedor_ruc_final", P, "proveedor_ruc_final"),
                              ("procesos_modelo.entidad_ruc", P, "entidad_ruc"),
                              ("dim_proveedor.ruc", DP, "ruc"), ("dim_entidad.ruc", DE, "ruc")]:
        filas = con.execute(f"SELECT length(CAST({col} AS VARCHAR)), count(*) FROM read_parquet('{q(ruta)}') "
                            f"WHERE {col} IS NOT NULL GROUP BY 1 ORDER BY 1").fetchall()
        print(f"     {nombre:<38}" + ", ".join(f"{l} caracteres: {miles(n)}" for l, n in filas))


seguro("4) UNIONES (todo en conteos)", parte4)

con.close()
print("\nFin de la celda. No se modificó ningún archivo.")

## E. Versión anterior del capítulo 1 (no correr)

Se conserva con sus salidas como evidencia. La primera celda termina con `BinderException` porque pide la columna `familia`, que en `planA_procesos_modelo` se llama `familia_procedimiento`; la segunda dependía de la primera. Las reemplaza la sección A.

In [ ]:
# =====================================================================================
# CAPÍTULO 1: tablas y figuras de contexto desde la base (solo lectura, sin puntajes ni SRI)
# =====================================================================================
import os, duckdb, pandas as pd, matplotlib.pyplot as plt

BASE = r"C:\SERCOP_FINAL\plan_A"
OUT = r"C:\SERCOP_FINAL\capitulo1"
os.makedirs(OUT, exist_ok=True)
PM = os.path.join(BASE, "planA_procesos_modelo.parquet").replace("\\", "/")
con = duckdb.connect()

# 1) Procesos y monto por año (año = inicio del período de ofertas). Serie hasta 2024.
cols = [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{PM}')").fetchall()]
anio = "anio_proceso" if "anio_proceso" in cols else "year(fecha_inicio_tender)"
por_anio = con.execute(f"""
    SELECT {anio} AS anio, COUNT(*) AS procesos, SUM(monto_final)/1e6 AS millones_usd
    FROM read_parquet('{PM}') WHERE {anio} BETWEEN 2015 AND 2024
    GROUP BY 1 ORDER BY 1""").df()
total = con.execute(f"SELECT COUNT(*), SUM(monto_final)/1e6 FROM read_parquet('{PM}')").fetchone()
print(f"Base completa: {total[0]:,} procesos | {total[1]:,.1f} millones USD")
print("\nTabla A. Procesos y monto adjudicado por año (2015-2024)")
print(por_anio.round(1).to_string(index=False))

# 2) Monto y procesos por familia de procedimiento (base completa)
por_familia = con.execute(f"""
    SELECT familia, COUNT(*) AS procesos, SUM(monto_final)/1e6 AS millones_usd,
           MEDIAN(monto_final) AS monto_mediano_usd
    FROM read_parquet('{PM}') GROUP BY 1 ORDER BY millones_usd DESC""").df()
por_familia["pct_monto"] = 100 * por_familia["millones_usd"] / por_familia["millones_usd"].sum()
print("\nTabla B. Procesos y monto por familia de procedimiento")
print(por_familia.round(1).to_string(index=False))

por_anio.to_csv(os.path.join(OUT, "tabla_A_por_anio.csv"), index=False, sep=";", decimal=",")
por_familia.to_csv(os.path.join(OUT, "tabla_B_por_familia.csv"), index=False, sep=";", decimal=",")

# Figura A: barras = procesos, línea = monto
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(por_anio["anio"].astype(str), por_anio["procesos"], color="#9aa5b1")
ax.set_ylabel("Procesos")
ax2 = ax.twinx()
ax2.plot(por_anio["anio"].astype(str), por_anio["millones_usd"], color="#1f5fbf", marker="o")
ax2.set_ylabel("Monto adjudicado (millones USD)")
ax.set_title("Procesos y monto adjudicado por año, 2015-2024")
for s in ["top"]:
    ax.spines[s].set_visible(False); ax2.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "figura_A_por_anio.png"), dpi=300)

# Figura B: monto por familia (barras horizontales)
fig, ax = plt.subplots(figsize=(9, 4.5))
t = por_familia.sort_values("millones_usd")
ax.barh(t["familia"], t["millones_usd"], color="#9aa5b1")
for y, v in enumerate(t["millones_usd"]):
    ax.text(v, y, f" {v:,.0f}", va="center", fontsize=8)
ax.set_xlabel("Monto adjudicado (millones USD)")
ax.set_title("Monto adjudicado por familia de procedimiento, 2015-2026")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "figura_B_por_familia.png"), dpi=300)
print("\nFiguras y tablas guardadas en", OUT)

In [ ]:
# 2) Monto y procesos por familia de procedimiento (base completa)
por_familia = con.execute(f"""
    SELECT familia_procedimiento AS familia, COUNT(*) AS procesos, SUM(monto_final)/1e6 AS millones_usd,
           MEDIAN(monto_final) AS monto_mediano_usd
    FROM read_parquet('{PM}') GROUP BY 1 ORDER BY millones_usd DESC""").df()
por_familia["pct_monto"] = 100 * por_familia["millones_usd"] / por_familia["millones_usd"].sum()
print("Tabla B. Procesos y monto por familia de procedimiento")
print(por_familia.round(1).to_string(index=False))

por_anio.to_csv(os.path.join(OUT, "tabla_A_por_anio.csv"), index=False, sep=";", decimal=",")
por_familia.to_csv(os.path.join(OUT, "tabla_B_por_familia.csv"), index=False, sep=";", decimal=",")

# Figura A: barras = procesos, línea = monto
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(por_anio["anio"].astype(str), por_anio["procesos"], color="#9aa5b1")
ax.set_ylabel("Procesos")
ax2 = ax.twinx()
ax2.plot(por_anio["anio"].astype(str), por_anio["millones_usd"], color="#1f5fbf", marker="o")
ax2.set_ylabel("Monto adjudicado (millones USD)")
ax.set_title("Procesos y monto adjudicado por año, 2015-2024")
ax.spines["top"].set_visible(False); ax2.spines["top"].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "figura_A_por_anio.png"), dpi=300)

# Figura B: monto por familia (barras horizontales)
fig, ax = plt.subplots(figsize=(9, 4.5))
t = por_familia.sort_values("millones_usd")
ax.barh(t["familia"].astype(str), t["millones_usd"], color="#9aa5b1")
for y, v in enumerate(t["millones_usd"]):
    ax.text(v, y, f" {v:,.0f}", va="center", fontsize=8)
ax.set_xlabel("Monto adjudicado (millones USD)")
ax.set_title("Monto adjudicado por familia de procedimiento, 2015-2026")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "figura_B_por_familia.png"), dpi=300)
print("\nFiguras y tablas guardadas en", OUT)

In [ ]:
# =====================================================================================
# PUNTO 7 · ACTUALIZAR EL PAQUETE DE COMPAÑEROS (plan_A\_envio_companeros), una sola vez
#   - Las versiones anteriores de indicadores, dimensión de entidad y especificación pasan a
#     _envio_companeros\_anterior_2026-10-25\ (no se borra nada).
#   - planA_validacion_sri sale del paquete: pasa a plan_A\_versiones\retirado_de_envio_companeros_2026-10-25\.
#   - Se copian los archivos vigentes y, en documentacion\, la nota fechada, enmiendas.md y el manifiesto
#     del 25 de octubre.
#   Antes de mover nada comprueba las huellas; si algo no es lo esperado, se detiene sin tocar nada.
#   No abre los datos ni imprime RUC: solo mueve, copia y calcula huellas.
# =====================================================================================
import os, shutil, hashlib

PA = r"C:\SERCOP_FINAL\plan_A"
ENV = os.path.join(PA, "_envio_companeros")
ANT = os.path.join(ENV, "_anterior_2026-10-25")
FUERA = os.path.join(PA, "_versiones", "retirado_de_envio_companeros_2026-10-25")

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

if os.path.exists(ANT) or os.path.exists(FUERA):
    raise SystemExit("Esta celda ya se corrió (existe la carpeta de anteriores o de retirados). No se repite.")

# (a) qué hay hoy y qué debería haber (16 primeros caracteres de la huella)
ESPERADO_VIEJO = {
    os.path.join(ENV, "planA_indicadores.parquet"): "ca8d7a8838da235c",          # v4_3, 60 columnas
    os.path.join(ENV, "planA_dim_entidad.parquet"): "eea23963f57a11eb",          # antes de la tipología
    os.path.join(ENV, "planA_procesos_modelo.parquet"): "adb1768ebd2dd83a",      # igual al vigente, se queda
    os.path.join(ENV, "planA_dim_proveedor.parquet"): "6ca961e876f1e5e5",        # igual al vigente, se queda
}
VIGENTE = {
    os.path.join(PA, "planA_indicadores.parquet"): "dc87f4b8e1282d7f",
    os.path.join(PA, "planA_dim_entidad.parquet"): "ab67e110578f1d45",
}
problemas = []
for p, h in {**ESPERADO_VIEJO, **VIGENTE}.items():
    if not os.path.exists(p):
        problemas.append(f"no existe {p}")
    elif sha(p)[:16] != h:
        problemas.append(f"{p}: huella {sha(p)[:16]}, se esperaba {h}")
DOC_V = os.path.join(PA, "documentacion")
copiar_doc = ["especificacion_indicadores.json", "nota_2026-10-25_especificacion_v6_2.md", "enmiendas.md",
              "manifiesto_2026-10-25.csv"]
for f in copiar_doc:
    if not os.path.exists(os.path.join(DOC_V, f)):
        problemas.append(f"falta {os.path.join(DOC_V, f)}")
if problemas:
    raise SystemExit("No se movió nada. Revisa:\n  " + "\n  ".join(problemas))
print("Huellas comprobadas: el paquete tiene lo que se esperaba y los archivos vigentes son los del preregistro.")

# (b) mover lo anterior (dentro del paquete, aparte) y retirar la validación del SRI (fuera del paquete)
os.makedirs(os.path.join(ANT, "documentacion"))
os.makedirs(FUERA)
mov = [(os.path.join(ENV, "planA_indicadores.parquet"), os.path.join(ANT, "planA_indicadores_v4_3.parquet")),
       (os.path.join(ENV, "planA_dim_entidad.parquet"), os.path.join(ANT, "planA_dim_entidad_previo_tipologia.parquet")),
       (os.path.join(ENV, "documentacion", "especificacion_indicadores.json"),
        os.path.join(ANT, "documentacion", "especificacion_indicadores_2026-10-09.json"))]
sri = os.path.join(ENV, "planA_validacion_sri.parquet")
if os.path.exists(sri):
    mov.append((sri, os.path.join(FUERA, "planA_validacion_sri.parquet")))
for a, b in mov:
    shutil.move(a, b)
    print(f"  movido   {os.path.relpath(a, PA)}  ->  {os.path.relpath(b, PA)}")

# (c) copiar lo vigente y comprobar que la copia es idéntica
cop = [(os.path.join(PA, "planA_indicadores.parquet"), os.path.join(ENV, "planA_indicadores.parquet")),
       (os.path.join(PA, "planA_dim_entidad.parquet"), os.path.join(ENV, "planA_dim_entidad.parquet"))]
cop += [(os.path.join(DOC_V, f), os.path.join(ENV, "documentacion", f)) for f in copiar_doc]
for a, b in cop:
    shutil.copy2(a, b)
    igual = sha(a) == sha(b)
    print(f"  copiado  {os.path.relpath(a, PA)}  ->  {os.path.relpath(b, PA)}  {'(idéntico)' if igual else 'DISTINTO'}")
    if not igual:
        raise SystemExit("Una copia no salió idéntica: avísame antes de enviar el paquete.")

# (d) cómo queda el paquete
print("\nPaquete de compañeros, primer nivel:")
for f in sorted(os.listdir(ENV)):
    p = os.path.join(ENV, f)
    if os.path.isfile(p):
        print(f"  {f:<36} {os.path.getsize(p)/1e6:8.1f} MB  {sha(p)[:16]}")
    else:
        print(f"  {f}\\")
print("\n¿Queda algún archivo del SRI dentro del paquete?",
      [os.path.relpath(os.path.join(d, f), ENV) for d, _, fs in os.walk(ENV) for f in fs
       if "validacion_sri" in f and f.endswith(".parquet")] or "no")
